# foldaudit — 2M-row test in Colab
Tests the **published** foldaudit 0.2.5 from PyPI on a 2,000,000-row dataset (NYC-taxi style: 3% positive label, 252 groups/zones, sorted timestamps).

**Expected results:** Test 1 OK, Test 2 FAIL (group leakage), Test 3 OK, Test 4 FAIL (time order). Each audit takes roughly 20 seconds on 2M rows.

In [ ]:
!pip install foldaudit
import foldaudit
print('foldaudit version:', foldaudit.__version__)

## 1. Build the 2M-row dataset

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from foldaudit import audit

n = 2_000_000
rng = np.random.default_rng(42)

# Label: 3% positive (like payment_type in the NYC taxi data)
y = (rng.random(n) < 0.03).astype(int)
# Groups: 252 zones (like pickup zones)
groups = rng.integers(0, 252, size=n)
# Time: sorted timestamps, one per second
times = pd.date_range('2024-01-01', periods=n, freq='s').to_numpy()
# X is only a placeholder: foldaudit audits the folds, it never trains a model
X = np.zeros((n, 1))

print(f'rows: {n:,}')
print(f'positive rate: {y.mean():.3%}')
print(f'groups (zones): {len(np.unique(groups))}')
print(f'time range: {times[0]} to {times[-1]}')

## 2. Test 1 — StratifiedKFold (expect OK)

In [ ]:
t0 = time.time()
rep = audit(StratifiedKFold(5, shuffle=True, random_state=42), X, y)
print(f'{time.time() - t0:.1f}s  verdict: {rep.verdict}')
print(rep.text)

## 3. Test 2 — KFold with groups (expect FAIL: group leakage)

In [ ]:
t0 = time.time()
rep = audit(KFold(5, shuffle=True, random_state=42), X, y, groups=groups)
print(f'{time.time() - t0:.1f}s  verdict: {rep.verdict}')
for f in rep.flags:
    print(f'[{f.severity}] {f.message}')

## 4. Test 3 — GroupKFold by zone (expect OK)

In [ ]:
t0 = time.time()
rep = audit(GroupKFold(5), X, y, groups=groups)
print(f'{time.time() - t0:.1f}s  verdict: {rep.verdict}')
print(rep.text)

## 5. Test 4 — KFold, no shuffle, with time (expect FAIL: time order)

In [ ]:
t0 = time.time()
rep = audit(KFold(5), X, y, time=times)
print(f'{time.time() - t0:.1f}s  verdict: {rep.verdict}')
for f in rep.flags:
    print(f'[{f.severity}] {f.message}')

## Done
If the verdicts match the expected results above, foldaudit 0.2.5 from PyPI is working correctly at 2M rows.